# CRSP 개별 종목 시험 (Colab)

노트북 RAM을 쓰지 않고 Colab에서 돌립니다. 위에서부터 셀을 차례로 실행하세요(▶). 끊기면 처음부터 다시 실행해도 끝난 부분은 건너뜁니다.

- 데이터: 내 드라이브의 `CRSP/` 폴더(원본 parquet). 결과도 같은 폴더 아래 `CRSP/lab_crsp/`에 저장되고, 노트북의 `H:\내 드라이브\CRSP\lab_crsp`로 자동 동기화됩니다.
- 런타임: 메뉴 **런타임 → 런타임 유형 변경**에서 CPU, 가능하면 **High-RAM**을 고르세요. GPU는 필요 없습니다.
- 규칙: `docs/research/trend_v2/DECISIONS.md` 70–71.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CRSP_DIR = '/content/drive/MyDrive/CRSP'        # 원본 CRSP_*.parquet
WORK_DIR = f'{CRSP_DIR}/lab_crsp'               # 패널·결과 (드라이브에 남음)
TOP_N = 1000                                     # 매년 말 시가총액 상위 N (DECISIONS 70)
BRANCH = 'claude/project-thread-6uqisi'          # 병합 뒤에는 'main'
os.makedirs(WORK_DIR, exist_ok=True)
print(sorted(f for f in os.listdir(CRSP_DIR) if f.endswith('.parquet')))

In [ ]:
%cd /content
!rm -rf trend-energy-etf-scanner
!git clone -q -b $BRANCH https://github.com/goodyyj617/trend-energy-etf-scanner.git
%cd trend-energy-etf-scanner
!pip install -q -r requirements.txt pyarrow
import multiprocessing, psutil
print(f'CPU {multiprocessing.cpu_count()}개 · RAM {psutil.virtual_memory().total / 1e9:.0f} GB')

## 1. 패널 만들기 (한 번만, 10–20분)
원본 7.7GB를 읽어 시점별 상위 N 패널을 `lab_crsp/panel_topN/`에 저장합니다. 이미 있으면 건너뜁니다.

In [ ]:
PANEL = f'{WORK_DIR}/panel_top{TOP_N}'
if os.path.exists(f'{PANEL}/meta.json'):
    print('패널이 이미 있습니다:', open(f'{PANEL}/meta.json').read())
else:
    !python -m lab.crsp --src "$CRSP_DIR" --out "$PANEL" --top $TOP_N --years 1991 2024

## 2. 시험 실행 (몇 시간, 중간 저장됨)
77개 계열 × 36칸 × 표본 2개, 그다음 Sobol 1,024점 × 표본 2개. 첫 계열이 끝나면 칸당 시간과 남은 예상 시간을 출력합니다.
브라우저 탭을 닫지 마세요(Pro는 백그라운드 실행이 없습니다). 끊기면 이 셀을 다시 실행하면 이어서 돕니다.

In [ ]:
OUT = f'{WORK_DIR}/study_top{TOP_N}'
WORKERS = max(1, min(4, multiprocessing.cpu_count() - 1, int(psutil.virtual_memory().total / 1e9 // 3)))
print('작업 프로세스', WORKERS)
!python -m lab.crsp_study --panel "$PANEL" --out "$OUT" --workers $WORKERS --sobol 1024

## 3. 끝나면
`lab_crsp/study_topN/lab_results/`에 앱용 연구 요약 2개와 시험 기록이 저장됩니다. 드라이브 동기화가 끝나면 스레드에 "끝났어"라고 알려 주세요. 노트북 쪽에서 결과를 앱의 **저장된 결과 → 연구 요약**에 넣고 메모를 씁니다.